# Nebula — audit and controlled story foundation experiment

Import as a **separate Kaggle notebook**. Enable Internet, a GPU and the `NEBULA_HF_TOKEN` write secret.
Stop any old training session after its confirmed save, then **Run all** here.

The fixed control is your saved **step-10,102 V18 model**. The experiment keeps its architecture, batch,
optimizer state and continued learning-rate schedule. It practices only a deterministic subset of existing
public TinyStories data: up to 8 MB training text and 0.5 MB validation text. This is a small foundation test,
not another full training campaign. No new dataset downloads or teacher API calls are needed;
it reads existing saved corpus shards from Hugging Face.

## What it checks

- Every document index: ranges, packed lengths, source totals, hash splits, duplicates and shard coverage.
- Sampled actual documents from every source/split: true BOS/EOS, UTF-8 integrity and matching content hashes.
- The original V18 batch at update 10,102: correct one-position input/target shift, lengths and token ranges.
- The source model on real batch text: future-token changes cannot affect past logits; manual next-token loss agrees.
- All selected story payloads and separation of their training/validation hashes.

Audit samples show actual public training text. Checks do not certify all corpus content, exclude near-duplicates,
or prove that earlier training never saw the validation texts. Validation is disjoint within the saved corpus.
Mixed windows beginning mid-word/UTF-8 character are expected for byte-level training.

## Controlled run

Baseline evaluation → **1,000 additional optimizer updates** → identical final evaluation.
Both evaluate held-out mixed text and held-out stories, greedy and seeded temperature-0.7/top-p-0.9 completions,
and ordinary chat prompts. Two prefixes come from validation stories. All replies remain visible for inspection.
Relevance/grammar require human judgment; lower loss or less repetition does not automatically approve the model.

One practical precision correction: a T4 uses native FP16 AMP with a gradient scaler, rather than emulated BF16.
The report records this. Because precision also changes if the source used emulated BF16, this is a diagnostic
experiment, not a perfectly isolated causal estimate of the data change. One GPU is used, even with T4 x2.
Expect roughly an hour of training at the previous speed, plus audits, baseline/final probes and uploads.
The training loop stops after two hours per session; setup/evaluation/uploads add time.

## Saves and resume

All datasets, model/optimizer checkpoints and reports stay in RAM and save directly to the PUBLIC repository
`iiidevv/nebula-raw-training`, under **experiments/nebula-v18-foundation-control/**.
Original V18 weights and manifest are preserved. Source documents are referenced from their existing saved shards.
No website update, automatic promotion, Google Drive, ZIP or local dataset/checkpoint files.
Only code, installed packages and the notebook use disk. Host RAM of 16 GB or more is recommended.

Saves every **100 updates** and on a graceful interrupt. Wait for `controlCheckpointConfirmed` before closing.
After a disconnect, Run all with identical settings/code to resume the last confirmed control checkpoint.
Completed runs display their saved comparison instead of training again. Changing settings requires a new OUTPUT_RUN.
Abrupt worker loss can lose RAM progress since the last confirmed save; public storage/history and GPU quotas still apply.


In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['HF_HUB_DISABLE_XET']='1'
subprocess.run([sys.executable,'-m','pip','install','-q',
    'huggingface_hub>=0.34,<2','requests>=2.32,<3','numpy>=1.26,<3',
    'pyarrow>=18,<24','fsspec>=2024.6,<2027'],check=True)
from huggingface_hub import constants
constants.HF_HUB_DISABLE_XET=True
# Keep the platform's existing PyTorch/CUDA installation (Runpod PyTorch 2.8 is suitable).
token=os.environ.get('NEBULA_HF_TOKEN') or os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('NEBULA_HF_TOKEN')
    except Exception:
        raise RuntimeError('Add and enable Kaggle secret NEBULA_HF_TOKEN, then Run all again.') from None
if not token or not token.strip():
    raise RuntimeError('Kaggle secret NEBULA_HF_TOKEN is empty.')
os.environ['NEBULA_HF_TOKEN']=token.strip()
del token


In [ ]:
OUTPUT_RUN='nebula-v18-foundation-control'
CONFIG={
    'sourceRepo':'iiidevv/nebula-raw-training',
    'sourceRun':'nebula-v18-raw-1gb',
    'sourceRevision':'e167fff3bb27c36c25460fcd4c61915a5b7016ed',
    'expectedUpdates':10102,
    'updates':1000,
    'trainUtf8Bytes':8_000_000,
    'validationUtf8Bytes':500_000,
    'seed':19021,
    'saveEvery':100,
    'maxSessionHours':2,
}


In [ ]:
sources={'nebula.py': '"""Nebula\'s small, randomly initialized causal Transformer. No pretrained weights."""\nimport json\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport torch\nfrom torch import nn\n\nBOS, EOS, SYSTEM, USER, ASSISTANT, PAD = range(256, 262)\nVOCAB = 262\nROLES = {"system": SYSTEM, "user": USER, "assistant": ASSISTANT}\n\n@dataclass\nclass Config:\n    width: int = 128\n    heads: int = 4\n    layers: int = 2\n    context: int = 512\n\ndef config_for_preset(preset):\n    presets = {\n        \'smoke\': Config(width=64,heads=2,layers=1,context=256),\n        \'cpu\': Config(),\n        \'cpu-medium\': Config(width=192,heads=6,layers=4,context=1024),\n        \'cpu-large\': Config(width=256,heads=8,layers=6,context=1024),\n        \'gpu\': Config(width=384,heads=8,layers=6,context=2048),\n    }\n    if preset not in presets: raise ValueError(\'Unknown model preset\')\n    return presets[preset]\n\ndef encode(messages, answer_prefix=False):\n    tokens, learn = [BOS], [False]\n    for message in messages:\n        role = message["role"]\n        tokens.append(ROLES[role]); learn.append(False)\n        content = list(message["content"].encode("utf-8"))\n        tokens.extend(content); learn.extend([role == "assistant"] * len(content))\n        tokens.append(EOS); learn.append(role == "assistant")\n    if answer_prefix:\n        tokens.append(ASSISTANT); learn.append(False)\n    return tokens, learn\n\ndef pack_prompt(messages, context):\n    systems = [m for m in messages if m[\'role\'] == \'system\']\n    history = [m for m in messages if m[\'role\'] != \'system\']\n    if not history or history[-1][\'role\'] != \'user\': raise ValueError(\'A user question is required\')\n    prefix, _ = encode(systems)\n    selected = []\n    # Keep every system instruction and the full latest question; never silently cut rules.\n    available = context - len(prefix) - 2\n    for message in reversed(history):\n        chunk = encode([message])[0][1:]\n        if len(chunk) > available:\n            if not selected: raise ValueError(\'Rules and question exceed the model context\')\n            break\n        selected.insert(0, chunk); available -= len(chunk)\n    return prefix + [token for chunk in selected for token in chunk] + [ASSISTANT]\n\nclass Nebula(nn.Module):\n    def __init__(self, config):\n        super().__init__(); self.config = config\n        self.token = nn.Embedding(VOCAB, config.width)\n        self.position = nn.Embedding(config.context, config.width)\n        self.blocks = nn.ModuleList(nn.TransformerEncoderLayer(config.width, config.heads, config.width * 4, dropout=0.0, activation="gelu", batch_first=True, norm_first=True) for _ in range(config.layers))\n        self.norm = nn.LayerNorm(config.width)\n        self.output = nn.Linear(config.width, VOCAB, bias=False)\n        self.output.weight = self.token.weight\n        self.register_buffer(\'_causal_mask\',torch.ones(config.context,config.context,dtype=torch.bool).triu(1),persistent=False)\n        self.apply(self._initialize)\n\n    @staticmethod\n    def _initialize(module):\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, std=0.02)\n            if isinstance(module, nn.Linear) and module.bias is not None: nn.init.zeros_(module.bias)\n\n    def forward(self, tokens):\n        length = tokens.shape[1]\n        if length > self.config.context: raise ValueError("Context is too long")\n        value = self.token(tokens) + self.position(torch.arange(length, device=tokens.device))\n        causal = self._causal_mask[:length,:length]\n        for block in self.blocks:\n            if self.training and getattr(self,\'_activation_checkpointing\',False) and torch.is_grad_enabled():\n                from torch.utils.checkpoint import checkpoint\n                value=checkpoint(block,value,src_mask=causal,use_reentrant=False)\n            else:value=block(value,src_mask=causal)\n        return self.output(self.norm(value))\n\n    @torch.inference_mode()\n    def reply(self, messages, max_tokens=160, temperature=0.7):\n        self.eval()\n        encoded = pack_prompt(messages, self.config.context)\n        device = next(self.parameters()).device\n        tokens = torch.tensor([encoded], device=device)\n        result = []\n        for _ in range(min(max_tokens, self.config.context - len(encoded))):\n            logits = self(tokens)[:, -1, :].clone()\n            end = logits[:, EOS].clone(); logits[:, BOS:] = float(\'-inf\'); logits[:, EOS] = end\n            if temperature == 0: next_token = logits.argmax(-1, keepdim=True)\n            else: next_token = torch.multinomial(torch.softmax(logits / temperature, -1), 1)\n            value = int(next_token.item())\n            if value == EOS: break\n            result.append(value); tokens = torch.cat([tokens, next_token], dim=1)\n        return bytes(result).decode(\'utf-8\', errors=\'replace\').strip()\n\ndef save(model, path, metadata, training_state=None):\n    if any(not torch.isfinite(parameter).all() for parameter in model.parameters()):\n        raise ValueError(\'Checkpoint contains non-finite weights; existing saved weights were not replaced\')\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(\'.tmp\')\n    payload={\'format\': 1, \'config\': asdict(model.config), \'state\': model.state_dict(), \'metadata\': metadata}\n    if training_state is not None: payload[\'trainingState\']=training_state\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\ndef load(path):\n    data = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if data.get(\'format\') != 1: raise ValueError(\'Unsupported model checkpoint\')\n    model = Nebula(Config(**data[\'config\'])); model.load_state_dict(data[\'state\']); model.eval()\n    if any(not torch.isfinite(parameter).all() for parameter in model.parameters()):\n        raise ValueError(\'Checkpoint contains non-finite weights; resume from a healthy best checkpoint\')\n    return model, data.get(\'metadata\', {})\n', 'v18_stream.py': '"""RAM-only raw corpus preparation and CUDA training; durable state lives on HF."""\nimport argparse\nfrom collections import OrderedDict\nimport copy\nimport fnmatch\nimport gc\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport random\nimport re\nimport time\nimport unicodedata\nfrom urllib.parse import quote\nimport numpy as np\nimport requests\nimport torch\nimport torch.nn.functional as F\nfrom nebula import Nebula,Config,BOS,EOS\nfrom v8_store import safe_server_detail\n\nFORMAT=\'nebula-v18-ram-stream-1\'\nSOURCES={\n \'fineweb\':{\'dataset\':\'HuggingFaceFW/fineweb-edu\',\'subset\':\'sample-10BT\',\'revision\':\'87f09149ef4734204d70ed1d046ddc9ca3f2b8f9\',\'license\':\'odc-by\'},\n \'cosmopedia\':{\'dataset\':\'HuggingFaceTB/smollm-corpus\',\'subset\':\'cosmopedia-v2\',\'revision\':\'3ba9d605774198c5868892d7a8deda78031a781f\',\'license\':\'odc-by\'},\n \'stories\':{\'dataset\':\'roneneldan/TinyStories\',\'subset\':\'default\',\'revision\':\'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64\',\'license\':\'cdla-sharing-1.0\'},\n \'textbooks\':{\'dataset\':\'HuggingFaceTB/cosmopedia\',\'subset\':\'openstax\',\'revision\':\'0ae6ec63f91742bd2d1eaef4f02232c55d719385\',\'license\':\'apache-2.0\'}}\nDEFAULT={\'sourceRepo\':\'iiidevv/nebula-research\',\'sourceRun\':\'nebula-v17-lab\',\'sourceKey\':\'current-foundationWeights\',\n \'textBytes\':1_000_000_000,\'weights\':{\'fineweb\':50,\'cosmopedia\':25,\'stories\':15,\'textbooks\':10},\n \'seed\':1802026,\'validationPercent\':2,\'minDocumentBytes\':200,\'maxDocumentBytes\':1_000_000,\n \'shardTextBytes\':32_000_000,\'steps\':100_000,\'batch\':8,\'microbatch\':1,\'learningRate\':0.00003,\n \'warmup\':200,\'cloudSaveEvery\':500,\'evaluateEvery\':1000,\'shardHoldSteps\':64}\nSECRET=re.compile(r\'\\b(?:hf_|gsk_|sk-|AIza|AQ\\.)[\\w-]{18,}|\\bBearer\\s+[\\w.~/+=-]{18,}\')\nPROBES=[\'A child found a small boat beside the pond.\',\'Ice melts when it takes in heat.\',\n        \'The rain stopped, and the neighbors opened their windows.\']\n\n\ndef emit(**value):print(json.dumps(value),flush=True)\ndef sha(value):return hashlib.sha256(value).hexdigest()\ndef digest(value):return sha(json.dumps(value,sort_keys=True,separators=(\',\',\':\')).encode())\ndef engine_hash():\n    # Source code is allowed on disk; corpus/model/optimizer artifacts never use local files.\n    from pathlib import Path\n    return sha((Path(__file__).read_text(encoding=\'utf-8\')+Path(__file__).with_name(\'nebula.py\').read_text(encoding=\'utf-8\')).encode())\n\n\ndef validate(config):\n    if set(config)!=set(DEFAULT):raise ValueError(\'Use exactly the documented CONFIG keys\')\n    if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',config[\'sourceRepo\']) or not re.fullmatch(r\'nebula-[\\w-]{1,55}\',config[\'sourceRun\']) or not re.fullmatch(r\'[a-zA-Z][\\w-]{0,90}\',config[\'sourceKey\']):\n        raise ValueError(\'Invalid source reference\')\n    bounds={\'textBytes\':(10000,4_000_000_000),\'seed\':(0,2**31),\'validationPercent\':(1,10),\n       \'minDocumentBytes\':(32,10000),\'maxDocumentBytes\':(1000,1_000_000),\'shardTextBytes\':(1000,64_000_000),\n       \'steps\':(1,1_000_000),\'batch\':(1,32),\'microbatch\':(1,8),\'warmup\':(1,5000),\n       \'cloudSaveEvery\':(1,5000),\'evaluateEvery\':(1,10000),\'shardHoldSteps\':(1,256)}\n    for key,(lo,hi) in bounds.items():\n        if type(config[key]) is not int or not lo<=config[key]<=hi:raise ValueError(\'Invalid \'+key)\n    mix=config[\'weights\']\n    if set(mix)!=set(SOURCES) or any(type(v) is not int or not 0<=v<=100 for v in mix.values()) or sum(mix.values())!=100:raise ValueError(\'Data percentages must total 100\')\n    if config[\'microbatch\']>config[\'batch\'] or config[\'minDocumentBytes\']>config[\'maxDocumentBytes\']:raise ValueError(\'Invalid batch/document limits\')\n    if type(config[\'learningRate\']) not in (int,float) or not math.isfinite(config[\'learningRate\']) or not 0<config[\'learningRate\']<=.001:raise ValueError(\'Invalid learning rate\')\n    return config\n\n\nclass Hub:\n    """Explicit byte transfers: no hf_hub_download, filesystem cache or temporary data files."""\n    def __init__(self,repo,token,run):\n        from huggingface_hub import HfApi\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or not re.fullmatch(r\'nebula-v18-[\\w-]{1,45}\',run):raise ValueError(\'Invalid public destination/run\')\n        self.repo=repo;self.token=token;self.prefix=\'experiments/\'+run+\'/\'\n        self.api=HfApi(token=token);self.session=requests.Session()\n        if self.api.whoami()[\'name\']!=repo.split(\'/\')[0]:raise ValueError(\'Use your own public output namespace\')\n        self.api.create_repo(repo,repo_type=\'model\',private=False,exist_ok=True)\n        info=self.api.model_info(repo)\n        if info.private is not False:raise ValueError(\'V18 output must be a separate public repository\')\n        self.revision=info.sha\n\n    def fetch(self,repo,path,revision,*,dataset=False):\n        prefix=\'datasets/\' if dataset else \'\'\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or \'..\' in path.split(\'/\') or path.startswith(\'/\') or \'\\\\\' in path:\n            raise ValueError(\'Invalid remote file reference\')\n        url=\'https://huggingface.co/\'+prefix+repo+\'/resolve/\'+quote(revision,safe=\'\')+\'/\'+quote(path,safe=\'/\')\n        for attempt in range(4):\n            response=self.session.get(url,headers={\'Authorization\':\'Bearer \'+self.token},timeout=(30,180))\n            if response.status_code in [408,429,500,502,503,504] and attempt<3:\n                response.close();time.sleep(min(60,5*2**attempt));continue\n            response.raise_for_status();return response.content\n        raise RuntimeError(\'Remote read failed\')\n\n    def state(self):\n        try:return json.loads(self.fetch(self.repo,self.prefix+\'state.json\',self.revision))\n        except requests.HTTPError as error:\n            if error.response.status_code==404:return None\n            raise\n\n    def read_entry(self,entry,repo=None,revision=None,prefix=None):\n        repo=repo or self.repo;revision=revision or self.revision;prefix=prefix or self.prefix\n        if type(entry.get(\'bytes\')) is not int or not 0<entry[\'bytes\']<=2_147_483_648:raise ValueError(\'Invalid object size\')\n        parts=entry.get(\'parts\',[entry])\n        if not 1<=len(parts)<=32 or sum(p[\'bytes\'] for p in parts)!=entry[\'bytes\']:raise ValueError(\'Invalid object parts\')\n        data=bytearray()\n        for part in parts:\n            if not part[\'path\'].startswith(prefix):raise ValueError(\'Unexpected object namespace\')\n            value=self.fetch(repo,part[\'path\'],revision)\n            if len(value)!=part[\'bytes\'] or sha(value)!=part[\'sha256\']:raise ValueError(\'Object part checksum mismatch\')\n            data.extend(value)\n        if sha(data)!=entry[\'sha256\']:raise ValueError(\'Object checksum mismatch\')\n        return data\n\n    def publish(self,state,blobs=None,root_readme=None):\n        from huggingface_hub import CommitOperationAdd\n        result=copy.deepcopy(state);operations=[];blobs=blobs or {}\n        for key,(name,blob) in blobs.items():\n            if \'..\' in name.split(\'/\') or name.startswith(\'/\') or \'\\\\\' in name:raise ValueError(\'Invalid output name\')\n            size=len(blob)\n            if not 0<size<=2_147_483_648:raise ValueError(\'Invalid output object size\')\n            identifier=sha(blob);parts=[]\n            for offset in range(0,size,64*1024*1024):\n                chunk=bytes(memoryview(blob)[offset:offset+64*1024*1024])\n                remote=self.prefix+\'objects/\'+identifier+\'/\'+name+f\'.part{len(parts):03d}.bin\'\n                parts.append({\'path\':remote,\'bytes\':len(chunk),\'sha256\':sha(chunk)})\n                operations.append(CommitOperationAdd(path_in_repo=remote,path_or_fileobj=chunk))\n            result[key]={\'bytes\':size,\'sha256\':identifier,\'parts\':parts}\n        # Raw object pointers added by the caller are resolved before this small atomic manifest.\n        for split in [\'train\',\'validation\',\'index\']:\n            key=\'_new_\'+split\n            if key in result:\n                result[\'corpus\'][split].append(result.pop(key))\n        operations.append(CommitOperationAdd(path_in_repo=self.prefix+\'state.json\',path_or_fileobj=json.dumps(result,indent=2).encode()))\n        if root_readme:operations.append(CommitOperationAdd(path_in_repo=\'README.md\',path_or_fileobj=root_readme.encode()))\n        for attempt in range(4):\n            try:\n                saved=self.api.create_commit(repo_id=self.repo,parent_commit=self.revision,operations=operations,commit_message=\'Save Nebula RAM-only raw experiment\')\n                break\n            except Exception as error:\n                response=getattr(error,\'response\',None);status=getattr(response,\'status_code\',None)\n                retry=status in [408,429,500,502,503,504] and attempt<3\n                emit(v18BackupFailed=True,httpStatus=status,serverDetail=safe_server_detail(error),retrying=retry)\n                if not retry:raise\n                time.sleep(min(60,5*2**attempt))\n        self.revision=saved.oid\n        if json.loads(self.fetch(self.repo,self.prefix+\'state.json\',self.revision))!=result:raise ValueError(\'Remote manifest verification failed\')\n        emit(v18PublicBackupConfirmed=True,revision=self.revision,uniqueTextBytes=result.get(\'uniqueTextBytes\',0),\n             updates=result.get(\'updates\',0),corpusComplete=result.get(\'corpusComplete\',False),publicWebsiteUpdated=False)\n        return result\n\n\ndef source_paths(hub,key):\n    spec=SOURCES[key];info=hub.api.dataset_info(spec[\'dataset\'],revision=spec[\'revision\'])\n    card=info.card_data.to_dict() if hasattr(info.card_data,\'to_dict\') else dict(info.card_data or {})\n    selected=next((c for c in card.get(\'configs\',[]) if c.get(\'config_name\')==spec[\'subset\']),None)\n    patterns=[]\n    if selected:\n        for item in selected.get(\'data_files\',[]):\n            if isinstance(item,dict) and item.get(\'split\')==\'train\':\n                value=item[\'path\'];patterns.extend(value if isinstance(value,list) else [value])\n    elif spec[\'subset\']==\'default\':patterns=[\'data/train-*.parquet\']\n    if not patterns:raise ValueError(\'Pinned source has no declared train-file pattern: \'+key)\n    paths=[name for name in hub.api.list_repo_files(spec[\'dataset\'],repo_type=\'dataset\',revision=spec[\'revision\'])\n           if name.endswith(\'.parquet\') and any(fnmatch.fnmatch(name,p) for p in patterns)]\n    if not paths:raise ValueError(\'No pinned parquet training files found: \'+key)\n    return sorted(paths)\n\n\ndef source_rows(hub,key,cursor,seed):\n    from huggingface_hub import HfFileSystem\n    import pyarrow.parquet as pq\n    spec=SOURCES[key];paths=source_paths(hub,key)\n    random.Random(seed+sum(key.encode())).shuffle(paths)\n    fs=HfFileSystem(token=hub.token)\n    for file_index in range(cursor[\'file\'],len(paths)):\n        remote=f"datasets/{spec[\'dataset\']}@{spec[\'revision\']}/{paths[file_index]}"\n        start=cursor[\'row\'] if file_index==cursor[\'file\'] else 0\n        # HfFileSystem\'s fsspec block cache lives in RAM. No local parquet/cache files.\n        with fs.open(remote,\'rb\',block_size=4*1024*1024,cache_type=\'bytes\') as stream:\n            parquet=pq.ParquetFile(stream);columns=[c for c in [\'text\',\'id\',\'url\',\'file_path\',\'audience\',\'format\'] if c in parquet.schema.names]\n            if \'text\' not in columns:raise ValueError(\'Pinned source is missing raw text: \'+key)\n            position=0\n            for group in range(parquet.num_row_groups):\n                size=parquet.metadata.row_group(group).num_rows\n                if position+size<=start:position+=size;continue\n                for batch in parquet.iter_batches(batch_size=32,row_groups=[group],columns=columns):\n                    for row in batch.to_pylist():\n                        index=position;position+=1\n                        if index<start:continue\n                        yield row,{\'file\':file_index,\'row\':index+1},paths[file_index]\n    # Exhaustion is checked against the requested source budget by prepare().\n\n\ndef convert(row,config):\n    text=row.get(\'text\')\n    if not isinstance(text,str):return None\n    text=text.replace(\'\\r\\n\',\'\\n\').replace(\'\\r\',\'\\n\').strip()\n    try:content=text.encode(\'utf-8\')\n    except UnicodeError:return None\n    if not config[\'minDocumentBytes\']<=len(content)<=config[\'maxDocumentBytes\']:return None\n    if \'\\ufffd\' in text or any(ord(c)<32 and c not in \'\\n\\t\' for c in text) or SECRET.search(text):return None\n    if any(p.casefold() in text.casefold() for p in PROBES):return None\n    identifier=sha(\' \'.join(unicodedata.normalize(\'NFKC\',text).casefold().split()).encode())\n    return content,identifier\n\n\ndef prepare(hub,state,hours,rows_fn=source_rows):\n    config=state[\'config\'];state=copy.deepcopy(state);seen=set()\n    for entry in state[\'corpus\'][\'index\']:\n        for line in gzip.decompress(hub.read_entry(entry)).splitlines():seen.add(json.loads(line)[\'hash\'])\n    buffers={\'train\':bytearray(),\'validation\':bytearray()};records=[];pending_bytes=0\n    started=time.monotonic();last_log=state[\'uniqueTextBytes\']\n    def flush():\n        nonlocal state,pending_bytes,records,buffers\n        blobs={}\n        for split in buffers:\n            if buffers[split]:blobs[\'_new_\'+split]=(split+\'.bin\',buffers[split])\n        if records:blobs[\'_new_index\']=(\'public-index.jsonl.gz\',gzip.compress(b\'\\n\'.join(json.dumps(r).encode() for r in records),mtime=0))\n        state=hub.publish(state,blobs)\n        buffers={\'train\':bytearray(),\'validation\':bytearray()};records=[];pending_bytes=0\n    try:\n        for key in SOURCES:\n            goal=config[\'textBytes\']*config[\'weights\'][key]//100\n            if state[\'sourceBytes\'][key]>=goal*.995:continue\n            emit(v18Acquiring=key,goalUniqueBytes=goal,resumeCursor=state[\'cursors\'][key])\n            for row,cursor,path in rows_fn(hub,key,state[\'cursors\'][key],config[\'seed\']):\n                if time.monotonic()-started>=hours*3600:flush();emit(v18PreparationPaused=True);return state\n                converted=convert(row,config)\n                state[\'cursors\'][key]=cursor\n                if converted:\n                    content,identifier=converted\n                    if identifier not in seen and len(content)<=goal-state[\'sourceBytes\'][key]:\n                        split=\'validation\' if int(identifier[:8],16)%100<config[\'validationPercent\'] else \'train\'\n                        tokens=np.empty(len(content)+2,dtype=\'<u2\');tokens[0]=BOS;tokens[-1]=EOS;tokens[1:-1]=np.frombuffer(content,dtype=np.uint8)\n                        offset=len(buffers[split])//2;buffers[split].extend(tokens.tobytes())\n                        provenance={k:str(row[k])[:1000] for k in [\'id\',\'url\',\'file_path\',\'audience\',\'format\'] if k in row}\n                        if SECRET.search(json.dumps(provenance)):raise ValueError(\'Credential pattern in upstream attribution\')\n                        records.append({\'hash\':identifier,\'source\':key,\'sourceFile\':path,\'sourceRow\':cursor[\'row\']-1,\n                            \'split\':split,\'shard\':len(state[\'corpus\'][split]),\'startToken\':offset,\'endToken\':len(buffers[split])//2,\n                            \'utf8Bytes\':len(content),\'provenance\':provenance})\n                        seen.add(identifier);state[\'sourceBytes\'][key]+=len(content);state[\'uniqueTextBytes\']+=len(content)\n                        state[\'documents\']+=1;pending_bytes+=len(content)\n                if state[\'uniqueTextBytes\']-last_log>=1_000_000:\n                    last_log=state[\'uniqueTextBytes\'];emit(v18PreparedBytes=last_log,source=key,documents=state[\'documents\'])\n                if pending_bytes>=config[\'shardTextBytes\']:flush()\n                if state[\'sourceBytes\'][key]>=goal*.995 or goal-state[\'sourceBytes\'][key]<config[\'minDocumentBytes\']:break\n            if state[\'sourceBytes\'][key]<goal*.90:flush();raise ValueError(\'Source exhausted below budget: \'+key)\n        state[\'corpusComplete\']=True;flush();return state\n    except KeyboardInterrupt:\n        flush();emit(v18PreparationPaused=True);return state\n\n\nclass RemotePool:\n    def __init__(self,hub,entries,context,hold=64):\n        self.hub=hub;self.entries=[e for e in entries if e[\'bytes\']//2>context];self.context=context;self.hold=hold\n        self.cache=OrderedDict()\n        if not self.entries:raise ValueError(\'Not enough packed tokens for a full context window\')\n        self.weights=[e[\'bytes\']//2-context for e in self.entries]\n    def data(self,index):\n        if index not in self.cache:\n            self.cache[index]=np.frombuffer(self.hub.read_entry(self.entries[index]),dtype=\'<u2\')\n            if len(self.cache)>2:self.cache.popitem(last=False)\n        self.cache.move_to_end(index);return self.cache[index]\n    def batch(self,step,batch,seed):\n        # Shard locality bounds network traffic; selection is weighted by valid start positions.\n        index=random.Random(seed+step//self.hold).choices(range(len(self.entries)),weights=self.weights,k=1)[0]\n        data=self.data(index);rng=random.Random(seed+step*7919);result=[]\n        for _ in range(batch):\n            start=rng.randrange(len(data)-self.context)\n            values=torch.from_numpy(np.array(data[start:start+self.context+1],dtype=np.int64))\n            result.append((values[:-1],values[1:]))\n        return result\n\n\ndef serialize(model,metadata,training=None):\n    if any(not torch.isfinite(p).all() for p in model.parameters()):raise ValueError(\'Non-finite weights cannot replace a healthy public checkpoint\')\n    stream=io.BytesIO()\n    payload={\'format\':1,\'config\':vars(model.config),\'state\':model.state_dict(),\'metadata\':metadata}\n    if training is not None:payload[\'trainingState\']=training\n    torch.save(payload,stream)\n    return stream\n\n\ndef deserialize(blob):\n    payload=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True)\n    if payload.get(\'format\')!=1:raise ValueError(\'Unsupported Nebula checkpoint\')\n    model=Nebula(Config(**payload[\'config\']));model.load_state_dict(payload[\'state\'])\n    if any(not torch.isfinite(p).all() for p in model.parameters()):raise ValueError(\'Non-finite source weights\')\n    return model,payload\n\n\n@torch.inference_mode()\ndef evaluate(model,pool):\n    emit(v18EvaluationPhase=\'heldout-loss-and-generation\')\n    device=next(model.parameters()).device;model.eval();total=0.;n=0\n    for i in range(24):\n        x,y=pool.batch(i,1,880019)[0];logits=model(x[None].to(device))\n        total+=float(F.cross_entropy(logits.flatten(0,1),y.to(device),reduction=\'sum\'));n+=len(y)\n        if (i+1)%8==0:emit(v18EvaluationLossBlocks=i+1,total=24)\n    raw=[]\n    for prefix in PROBES:\n        tokens=[BOS]+list(prefix.encode())\n        for _ in range(128):\n            value=int(model(torch.tensor([tokens[-model.config.context:]],device=device))[0,-1].argmax())\n            if value>=256:break\n            tokens.append(value)\n        raw.append({\'prefix\':prefix,\'continuation\':bytes(t for t in tokens[1+len(prefix.encode()):] if t<256).decode(\'utf-8\',errors=\'replace\'),\'grade\':\'pending-human-review\'})\n        emit(v18RawProbesComplete=len(raw),total=3)\n    chat=[]\n    for prompt in [\'Hello! What can you help me with?\',\'Explain why ice melts.\',\'Write a thank-you note to a neighbor.\',\'Which should I choose?\']:\n        answer=model.reply([{\'role\':\'system\',\'content\':\'You are Nebula AI.\'},{\'role\':\'user\',\'content\':prompt}],max_tokens=128,temperature=0)\n        chat.append({\'prompt\':prompt,\'reply\':answer,\'grade\':\'pending-human-review\'})\n        emit(v18ChatProbesComplete=len(chat),total=4)\n    return {\'validationLoss\':total/n,\'validationTargets\':n,\'rawCompletions\':raw,\'openAnswers\':chat,\n            \'qualityApproved\':False,\'humanVerified\':False,\'publicWebsiteUpdated\':False}\n\n\ndef train(hub,state,hours,action,device=\'cuda\'):\n    config=state[\'config\'];state=copy.deepcopy(state)\n    if not state[\'corpusComplete\']:raise ValueError(\'Finish ACTION=prepare before using the GPU\')\n    if device==\'cuda\' and not torch.cuda.is_available():raise RuntimeError(\'CUDA GPU not detected. No CPU training fallback was started.\')\n    if not state.get(\'source\'):\n        info=hub.api.model_info(config[\'sourceRepo\'])\n        prefix=\'experiments/\'+config[\'sourceRun\']+\'/\'\n        old=json.loads(hub.fetch(config[\'sourceRepo\'],prefix+\'state.json\',info.sha))\n        entry=old.get(config[\'sourceKey\'])\n        if not entry:raise ValueError(\'Source checkpoint unavailable; no random restart was performed\')\n        model,payload=deserialize(hub.read_entry(entry,config[\'sourceRepo\'],info.sha,prefix));del payload\n        stream=serialize(model,{\'experimental\':True,\'qualityApproved\':False,\'originWeightSha256\':entry[\'sha256\']})\n        try:state=hub.publish({**state,\'sourceSha256\':entry[\'sha256\']},{\'source\':(\'source.pt\',stream.getbuffer())})\n        finally:stream.close()\n    else:model,payload=deserialize(hub.read_entry(state.get(\'working\') or state[\'source\']))\n    model.to(device);torch.set_num_threads(2)\n    dtype=torch.bfloat16 if device==\'cuda\' and torch.cuda.is_bf16_supported() else torch.float16\n    amp=device==\'cuda\';scaler=torch.amp.GradScaler(\'cuda\',enabled=amp and dtype==torch.float16,init_scale=1024)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=config[\'learningRate\'],weight_decay=.01)\n    signature=digest({\'config\':config,\'engine\':state[\'engineSha256\'],\'corpus\':state[\'corpus\'],\'source\':state[\'sourceSha256\']})\n    updates=targets=0\n    if state.get(\'working\'):\n        training=payload[\'trainingState\']\n        if training[\'signature\']!=signature:raise ValueError(\'Checkpoint/config mismatch; use a new run name for changes\')\n        optimizer.load_state_dict(training[\'optimizer\'])\n        if scaler.is_enabled() and training[\'scaler\']:scaler.load_state_dict(training[\'scaler\'])\n        elif scaler.is_enabled():emit(v18ScalerReset=True,reason=\'Source precision used no gradient scaler; optimizer state retained\')\n        updates=training[\'updates\'];targets=training[\'targets\'];del payload,training\n    elif \'payload\' in locals():del payload\n    pool=RemotePool(hub,state[\'corpus\'][\'train\'],model.config.context,config[\'shardHoldSteps\'])\n    validation=RemotePool(hub,state[\'corpus\'][\'validation\'],model.config.context,1)\n    emit(v18TrainingDevice=torch.cuda.get_device_name(0) if amp else \'CPU test\',parameters=sum(p.numel() for p in model.parameters()),\n         optimizerRestored=bool(state.get(\'working\')),uniqueCorpusBytes=state[\'uniqueTextBytes\'],precision=str(dtype) if amp else \'fp32\',\n         plannedTrainingBytePositions=config[\'steps\']*config[\'batch\']*model.config.context)\n    def report(label):\n        nonlocal state\n        result=evaluate(model,validation)\n        state=hub.publish({**state,\'latestEvaluation\':{\'label\':label,\'validationLoss\':result[\'validationLoss\']}},\n                          {\'evaluation\':(\'evaluation.json\',json.dumps(result).encode())})\n        emit(v18Evaluation=result)\n    if action==\'evaluate\':report(\'manual-\'+str(updates));return state\n    if action==\'train\' and not state.get(\'baselineEvaluated\'):\n        report(\'baseline\');state=hub.publish({**state,\'baselineEvaluated\':True})\n    started=time.monotonic();start_targets=targets;start_updates=updates;last_backup=updates;bad=0\n    budget=min(config[\'steps\'],updates+100) if action==\'benchmark\' else config[\'steps\']\n    def backup():\n        nonlocal state,last_backup\n        if updates==last_backup and state.get(\'working\'):return\n        stream=serialize(model,{\'updates\':updates,\'supervisedBytePositions\':targets,\'qualityApproved\':False,\'experimental\':True},\n              {\'signature\':signature,\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\'updates\':updates,\'targets\':targets})\n        try:state=hub.publish({**state,\'updates\':updates,\'trainingBytePositions\':targets}, {\'working\':(\'working.pt\',stream.getbuffer())})\n        finally:stream.close()\n        last_backup=updates\n    interrupted=False\n    try:\n        while updates<budget and time.monotonic()-started<hours*3600:\n            pairs=pool.batch(updates,config[\'batch\'],config[\'seed\']);model.train();optimizer.zero_grad(set_to_none=True)\n            rate=config[\'learningRate\']*min(1.,(updates+1)/config[\'warmup\'])*(.1+.9*.5*(1+math.cos(math.pi*updates/config[\'steps\'])))\n            for group in optimizer.param_groups:group[\'lr\']=rate\n            loss_value=0.;denominator=sum(len(y) for _,y in pairs)\n            for offset in range(0,len(pairs),config[\'microbatch\']):\n                micro=pairs[offset:offset+config[\'microbatch\']]\n                x=torch.stack([p[0] for p in micro]).to(device);y=torch.stack([p[1] for p in micro]).to(device)\n                with torch.autocast(\'cuda\',dtype=dtype,enabled=amp):\n                    objective=F.cross_entropy(model(x).flatten(0,1),y.flatten(),reduction=\'sum\')/denominator\n                if not torch.isfinite(objective):raise ValueError(\'Non-finite loss; previous public checkpoint retained\')\n                loss_value+=float(objective.detach());scaler.scale(objective).backward()\n            scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            if not torch.isfinite(norm):\n                bad+=1\n                if not scaler.is_enabled() or bad>12:raise ValueError(\'Non-finite gradients; previous checkpoint retained\')\n                scaler.update(new_scale=scaler.get_scale()/2);continue\n            bad=0;scaler.step(optimizer);scaler.update();updates+=1;targets+=denominator\n            if updates%25==0:emit(v18Update=updates,budget=config[\'steps\'],loss=round(loss_value,4),learningRate=rate,\n                supervisedBytePositions=targets,bytePositionsPerSecond=round((targets-start_targets)/max(1,time.monotonic()-started)))\n            if action==\'train\' and updates%config[\'cloudSaveEvery\']==0:\n                backup()\n                if updates%config[\'evaluateEvery\']==0:report(\'update-\'+str(updates))\n    except KeyboardInterrupt:interrupted=True\n    if action==\'benchmark\':\n        elapsed=time.monotonic()-started;rate=(targets-start_targets)/max(.01,elapsed)\n        result={\'benchmarkUpdates\':updates-start_updates,\'seconds\':elapsed,\'bytePositionsPerSecond\':rate,\n                \'estimatedComputeHoursForRemainingBudget\':(config[\'steps\']-start_updates)*config[\'batch\']*model.config.context/max(1,rate)/3600,\n                \'weightsDiscarded\':True,\'note\':\'Includes corpus downloads; excludes remote full-checkpoint saves and evaluation.\'}\n        state=hub.publish(state,{\'benchmark\':(\'benchmark.json\',json.dumps(result).encode())});emit(v18Benchmark=result)\n    else:\n        backup();report(\'session-\'+str(updates))\n        state=hub.publish({**state,\'trainingComplete\':updates>=config[\'steps\']})\n    emit(v18SessionFinished=True,interrupted=interrupted,updatesSaved=state.get(\'updates\',0),publicWebsiteUpdated=False)\n    del optimizer,model,pool,validation;gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    return state\n\n\ndef readme(repo):\n    attribution=\'\\n\'.join(f"- {key}: [{s[\'dataset\']}](https://huggingface.co/datasets/{s[\'dataset\']}/tree/{s[\'revision\']}), `{s[\'subset\']}`, `{s[\'license\']}`." for key,s in SOURCES.items())\n    return f\'\'\'---\nlicense: other\nlicense_name: per-source-data-licenses\nlicense_link: https://huggingface.co/{repo}/blob/main/README.md#licenses\nlibrary_name: pytorch\ntags:\n- nebula\n- experimental\n---\n# Nebula V18 — raw text and GPU experiment\n\nPublic experimental continuation of the owner\'s Nebula weights. Durable datasets, packed byte-token shards,\nsource indexes, model/optimizer/scaler checkpoints and reports are uploaded directly from RAM.\nNo dataset or checkpoint is written to the worker\'s filesystem. Python code, installed packages and\nthe notebook itself can exist on disk. This runner does not ingest account chats or review feeds.\nInherited weights and public web data are not guaranteed free of personal information.\nNo website deployment or automatic quality approval is enabled.\n\n## Licenses\n\nEach corpus retains its upstream terms and attribution; this collection does not relicense it.\nPinned upstream cards are stored with each run; original FineWeb document URLs are retained in compressed indexes.\nThe new textbooks component is synthetic Cosmopedia text seeded from OpenStax outlines.\nODC-By: https://opendatacommons.org/licenses/by/1-0/\nCDLA-Sharing-1.0: https://cdla.dev/sharing-1-0/\nApache-2.0: https://www.apache.org/licenses/LICENSE-2.0\n\n{attribution}\n\n## Representation and limitations\n\nPacked tokens are little-endian uint16: byte IDs 0–255, BOS=256, EOS=257. Whole UTF-8 documents\nare normalized-exact-deduplicated and assigned by hash to train or validation before packing.\nCredential-pattern/control/size filters are applied; this is not a full PII, factual or fuzzy-duplicate audit.\nThe model may have seen overlapping public text before this experiment.\nSource mix percentages describe acquired UTF-8 bytes; training samples packed positions, including repeated exposure.\nTraining batches use deterministic shard locality (64 updates per sampled shard) to bound network traffic.\nAt most two training shards and two validation shards are cached in RAM; deduplication hashes also use RAM.\nFull optimizer snapshots consume substantial host RAM and public storage; GPU memory alone is not sufficient.\nOnly confirmed public checkpoints survive worker loss. The history is retained and public storage is not unlimited.\n\'\'\'\n\n\ndef run(repo=\'iiidevv/nebula-raw-training\',run_name=\'nebula-v18-raw-1gb\',config=None,action=\'prepare\',hours=2,\n        *,hub_factory=Hub,rows_fn=source_rows,device=\'cuda\'):\n    config=validate(copy.deepcopy(DEFAULT if config is None else config))\n    if action not in [\'prepare\',\'benchmark\',\'train\',\'evaluate\'] or not 0<hours<=12:raise ValueError(\'Invalid action/session hours\')\n    if action!=\'prepare\' and device==\'cuda\' and not torch.cuda.is_available():raise RuntimeError(\'CUDA GPU not detected; no CPU training fallback started\')\n    if (repo,run_name)==(config[\'sourceRepo\'],config[\'sourceRun\']):raise ValueError(\'Do not overwrite the source experiment\')\n    token=os.environ.get(\'NEBULA_HF_TOKEN\') or os.environ.get(\'HF_TOKEN\')\n    if not token:raise RuntimeError(\'Set NEBULA_HF_TOKEN in the platform secret/environment settings\')\n    hub=hub_factory(repo,token,run_name);state=hub.state();code=engine_hash()\n    if state is None:\n        state={\'format\':FORMAT,\'config\':config,\'engineSha256\':code,\'sources\':SOURCES,\'uniqueTextBytes\':0,\'documents\':0,\n          \'sourceBytes\':{k:0 for k in SOURCES},\'cursors\':{k:{\'file\':0,\'row\':0} for k in SOURCES},\n          \'corpus\':{\'train\':[],\'validation\':[],\'index\':[]},\'corpusComplete\':False,\'updates\':0,\'qualityApproved\':False,\'publicAutoUpdate\':False}\n        blobs={}\n        for key,spec in SOURCES.items():blobs[\'card-\'+key]=(key+\'-README.md\',hub.fetch(spec[\'dataset\'],\'README.md\',spec[\'revision\'],dataset=True))\n        state=hub.publish(state,blobs,root_readme=readme(repo))\n    if state[\'format\']!=FORMAT or state[\'config\']!=config or state[\'sources\']!=SOURCES or state[\'engineSha256\']!=code:\n        raise ValueError(\'Run settings/code changed. Keep the original settings to resume or use a NEW RUN_NAME\')\n    if action==\'prepare\':\n        if not state[\'corpusComplete\']:state=prepare(hub,state,hours,rows_fn)\n        emit(v18DataReady=state[\'corpusComplete\'],uniqueUtf8Bytes=state[\'uniqueTextBytes\'],trainingStarted=False)\n        return state\n    return train(hub,state,hours,action,device)\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--action\',choices=[\'prepare\',\'benchmark\',\'train\',\'evaluate\'],default=\'prepare\')\n    parser.add_argument(\'--repo\',default=\'iiidevv/nebula-raw-training\');parser.add_argument(\'--run-name\',default=\'nebula-v18-raw-1gb\')\n    parser.add_argument(\'--hours\',type=float,default=2)\n    args=parser.parse_args();run(args.repo,args.run_name,action=args.action,hours=args.hours)\n\n\nif __name__==\'__main__\':main()\n', 'v8_store.py': '"""Atomic private manifests for large, sharded training checkpoints."""\nimport json\nfrom pathlib import Path\nimport shutil\nimport tempfile\nimport time\nimport os\nimport re\nfrom v4_store import Store as SmallStore\nfrom research_client import ResearchError,digests\n\nPART_BYTES=64*1024*1024\nMAX_BYTES=2*1024*1024*1024\n\ndef safe_server_detail(error):\n    """Expose the service\'s explanation, with credentials and URLs removed."""\n    detail=getattr(error,\'server_message\',None)\n    if not isinstance(detail,str) or not detail.strip():\n        try:\n            body=error.response.json()\n            detail=body.get(\'error\') or body.get(\'message\') if isinstance(body,dict) else None\n        except Exception:detail=None\n    if not isinstance(detail,str):return \'No structured server explanation was provided.\'\n    for key,value in os.environ.items():\n        if re.search(r\'TOKEN|SECRET|PASSWORD|API_KEY\',key,re.I) and len(value)>=6:\n            detail=detail.replace(value,\'[redacted]\')\n    detail=re.sub(r\'https?://\\S+\',\'[URL removed]\',detail)\n    detail=re.sub(r\'(?i)\\b(?:hf_|gsk_|AIza)[A-Za-z0-9_.-]+\',\'[redacted]\',detail)\n    detail=re.sub(r\'(?i)(?:bearer\\s+|(?:token|api[_-]?key|password|secret)\\s*[:=]\\s*)[^\\s,;]+\',\'[redacted]\',detail)\n    return \' \'.join(detail.split())[:700]\n\nclass Store(SmallStore):\n    def _commit_backup(self, operations):\n        """Retry temporary failures with the SAME parent; never overwrite another writer."""\n        for attempt in range(4):\n            try:\n                return self.api.create_commit(repo_id=self.repo,operations=operations,\n                    commit_message=\'Save Nebula 100M experiment\',parent_commit=self.revision)\n            except Exception as error:\n                response=getattr(error,\'response\',None)\n                status=getattr(response,\'status_code\',None)\n                # Do not expose exception text, response bodies, URLs, headers or credentials.\n                status=status if type(status) is int else None\n                network=any(kind.__name__ in {\'Timeout\',\'TimeoutError\',\'ConnectTimeout\',\'ReadTimeout\',\n                    \'ConnectionError\',\'ConnectError\',\'ReadError\',\'RemoteProtocolError\'} for kind in type(error).__mro__)\n                retryable=status in {408,429,500,502,503,504} or network\n                reason=(\'CONCURRENT_OR_AMBIGUOUS_COMMIT\' if status in {409,412} else\n                        \'ACCESS_DENIED\' if status in {401,403} else\n                        \'REQUEST_REJECTED\' if status in {400,413,422} else\n                        \'TEMPORARY_SERVICE_FAILURE\' if retryable else \'UNCLASSIFIED_FAILURE\')\n                retry=retryable and attempt<3\n                wait=5*(2**attempt)\n                if status==429:\n                    try:wait=max(wait,min(60,max(0,float(response.headers.get(\'Retry-After\',0)))))\n                    except (AttributeError,TypeError,ValueError):pass\n                print(json.dumps({\'backupFailed\':True,\'reason\':reason,\'httpStatus\':status,\n                    \'attempt\':attempt+1,\'retrying\':retry,\'retryInSeconds\':wait if retry else 0,\n                    \'serverDetail\':safe_server_detail(error),\n                    \'localFilesPreserved\':True}),flush=True)\n                if not retry:\n                    raise ResearchError(\'V8_BACKUP_FAILED_OR_CONCURRENT_RUN:\'+reason) from None\n                time.sleep(wait)\n\n    def download(self,file,destination):\n        if \'parts\' not in file:return super().download(file,destination)\n        if not 0<file.get(\'bytes\',0)<=MAX_BYTES or not isinstance(file.get(\'parts\'),list) or not 1<=len(file[\'parts\'])<=32 or sum(p.get(\'bytes\',0) for p in file[\'parts\'])!=file[\'bytes\']:\n            raise ResearchError(\'V8_INVALID_SHARDED_CHECKPOINT\')\n        destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)\n        if destination.is_file() and destination.stat().st_size==file[\'bytes\'] and digests(destination)[0]==file[\'sha256\']:return destination\n        temporary=destination.with_suffix(destination.suffix+\'.download-tmp\')\n        try:\n            with tempfile.TemporaryDirectory(dir=destination.parent,prefix=\'v8-parts-\') as folder:\n                with temporary.open(\'wb\') as output:\n                    for i,part in enumerate(file[\'parts\']):\n                        local=super().download(part,Path(folder)/str(i))\n                        with local.open(\'rb\') as source:shutil.copyfileobj(source,output,1024*1024)\n            if temporary.stat().st_size!=file[\'bytes\'] or digests(temporary)[0]!=file[\'sha256\']:raise ResearchError(\'V8_CHECKSUM_MISMATCH\')\n            temporary.replace(destination)\n        except BaseException:\n            temporary.unlink(missing_ok=True);raise\n        return destination\n\n    def publish(self,state,files,project,manifest_name=\'state.json\'):\n        from huggingface_hub import CommitOperationAdd\n        if manifest_name!=\'state.json\':raise ResearchError(\'V8_INVALID_MANIFEST\')\n        self.info();value=json.loads(json.dumps(state));operations=[]\n        with tempfile.TemporaryDirectory(dir=project,prefix=\'v8-upload-\') as folder:\n            for key,(path,name) in files.items():\n                path=Path(path);size=path.stat().st_size\n                if \'..\' in name.split(\'/\') or name.startswith(\'/\') or \'\\\\\' in name or not 0<size<=MAX_BYTES:raise ResearchError(\'V8_INVALID_FILE_REFERENCE\')\n                # Use a recognized binary extension for SQLite snapshots and shards.\n                # Local file names and existing manifests remain valid.\n                if name.endswith(\'.sqlite\'):name+=\'.bin\'\n                if size<=PART_BYTES:\n                    remote=self.prefix+name\n                    value[key]={\'path\':remote,\'bytes\':size,\'sha256\':digests(path)[0]}\n                    operations.append(CommitOperationAdd(path_in_repo=remote,path_or_fileobj=str(path)));continue\n                parts=[]\n                with path.open(\'rb\') as source:\n                    index=0\n                    while chunk:=source.read(PART_BYTES):\n                        local=Path(folder)/(key+\'-\'+str(index));local.write_bytes(chunk)\n                        remote=self.prefix+name+f\'.part{index:03d}.bin\'\n                        parts.append({\'path\':remote,\'bytes\':len(chunk),\'sha256\':digests(local)[0]})\n                        operations.append(CommitOperationAdd(path_in_repo=remote,path_or_fileobj=str(local)));index+=1\n                value[key]={\'bytes\':size,\'sha256\':digests(path)[0],\'parts\':parts}\n            manifest=Path(folder)/\'state.json\';manifest.write_text(json.dumps(value,indent=2),encoding=\'utf-8\')\n            operations.append(CommitOperationAdd(path_in_repo=self.prefix+manifest_name,path_or_fileobj=str(manifest)))\n            result=self._commit_backup(operations)\n        self.revision=result.oid\n        print(json.dumps({\'v8BackupSaved\':True,\'round\':value.get(\'round\',0),\'publicWebsiteUpdated\':False}),flush=True)\n        return value\n', 'v4_store.py': '"""Private experiment snapshots, committed atomically with a resume manifest."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nfrom research_client import ResearchError,digests\n\nclass Store:\n    def __init__(self,repo,token,run):\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or not re.fullmatch(r\'[a-zA-Z0-9_-]{1,64}\',run) or not token.startswith(\'hf_\'):\n            raise ResearchError(\'V4_STORAGE_CONFIGURATION_MISSING\')\n        from huggingface_hub import HfApi\n        self.repo=repo;self.token=token;self.prefix=\'experiments/\'+run+\'/\'\n        self.api=HfApi(endpoint=\'https://huggingface.co\',token=token)\n        self.revision=self.info().sha\n    def info(self):\n        try:value=self.api.model_info(self.repo)\n        except Exception:raise ResearchError(\'V4_STORAGE_ACCESS_FAILED\') from None\n        if value.private is not True:raise ResearchError(\'HF_REPOSITORY_MUST_BE_PRIVATE\')\n        return value\n    def state(self,manifest_name=\'state.json\'):\n        if manifest_name not in [\'state.json\',\'prepared.json\']:raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n        from huggingface_hub import hf_hub_download\n        from huggingface_hub.errors import EntryNotFoundError\n        try:\n            path=hf_hub_download(self.repo,self.prefix+manifest_name,revision=self.revision,token=self.token)\n            return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n        except EntryNotFoundError:return None\n        except Exception:raise ResearchError(\'V4_RESUME_MANIFEST_UNREADABLE\') from None\n    def download(self,file,destination):\n        from huggingface_hub import hf_hub_download\n        if not file[\'path\'].startswith(self.prefix) or \'..\' in file[\'path\'].split(\'/\') or not 0<file[\'bytes\']<=256*1024*1024:\n            raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n        destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)\n        if destination.exists() and destination.stat().st_size==file[\'bytes\'] and digests(destination)[0]==file[\'sha256\']:return destination\n        try:cached=Path(hf_hub_download(self.repo,file[\'path\'],revision=self.revision,token=self.token))\n        except Exception:raise ResearchError(\'V4_SNAPSHOT_DOWNLOAD_FAILED\') from None\n        if cached.stat().st_size!=file[\'bytes\'] or digests(cached)[0]!=file[\'sha256\']:raise ResearchError(\'V4_CHECKSUM_MISMATCH\')\n        temporary=destination.with_suffix(destination.suffix+\'.download-tmp\');shutil.copyfile(cached,temporary);temporary.replace(destination)\n        return destination\n    def publish(self,state,files,project,manifest_name=\'state.json\'):\n        if manifest_name not in [\'state.json\',\'prepared.json\']:raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n        from huggingface_hub import CommitOperationAdd\n        self.info();operations=[];state=json.loads(json.dumps(state))\n        for key,(path,name) in files.items():\n            path=Path(path)\n            if \'..\' in name.split(\'/\') or name.startswith(\'/\') or \'\\\\\' in name:raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n            if not 0<path.stat().st_size<=256*1024*1024:raise ResearchError(\'V4_SNAPSHOT_TOO_LARGE\')\n            remote=self.prefix+name\n            state[key]={\'path\':remote,\'bytes\':path.stat().st_size,\'sha256\':digests(path)[0]}\n            operations.append(CommitOperationAdd(path_in_repo=remote,path_or_fileobj=str(path)))\n        manifest=Path(project)/(\'v4-\'+manifest_name);manifest.write_text(json.dumps(state,indent=2),encoding=\'utf-8\')\n        operations.append(CommitOperationAdd(path_in_repo=self.prefix+manifest_name,path_or_fileobj=str(manifest)))\n        try:\n            result=self.api.create_commit(repo_id=self.repo,operations=operations,\n                commit_message=\'Save bounded Nebula V4 experiment\',parent_commit=self.revision)\n        except Exception:raise ResearchError(\'V4_BACKUP_FAILED_OR_CONCURRENT_RUN\') from None\n        self.revision=result.oid\n        print(json.dumps({\'v4BackupSaved\':True,\'round\':state.get(\'round\',0),\'publicWebsiteUpdated\':False}),flush=True)\n        return state\n', 'research_client.py': '"""Private worker protocol. Tokens never go to object-storage URLs or logs."""\nimport base64\nimport hashlib\nimport os\nimport re\nimport shutil\nfrom pathlib import Path\nfrom urllib.parse import urlparse\nimport requests\n\nclass ResearchError(RuntimeError):\n    def __init__(self,code,status=0):super().__init__(str(code));self.code=code;self.status=status\n\ndef digests(path):\n    sha=hashlib.sha256();md5=hashlib.md5(usedforsecurity=False)\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(1024*1024),b\'\'):sha.update(chunk);md5.update(chunk)\n    return sha.hexdigest(),base64.b64encode(md5.digest()).decode()\n\ndef storage_url(url):\n    parsed=urlparse(url)\n    if parsed.scheme!=\'https\' or parsed.username or parsed.password or not (\n            parsed.hostname==\'storage.googleapis.com\' or (parsed.hostname or \'\').endswith(\'.storage.googleapis.com\')):\n        raise ResearchError(\'UNEXPECTED_STORAGE_HOST\')\n    return url\n\nclass Client:\n    def __init__(self,api=None,token=None):\n        self.api=(api or os.environ.get(\'NEBULA_RESEARCH_API\',\'\')).rstrip(\'/\')\n        self.token=token or os.environ.get(\'NEBULA_WORKER_TOKEN\',\'\')\n        parsed=urlparse(self.api)\n        if parsed.scheme!=\'https\' or parsed.username or parsed.password or parsed.query or parsed.fragment or len(self.token)<32:\n            raise ResearchError(\'WORKER_CONFIGURATION_MISSING\')\n    def call(self,action,body=None):\n        try:\n            response=requests.request(\'GET\' if body is None else \'POST\',self.api+\'/\'+action,\n                headers={\'Authorization\':\'Bearer \'+self.token},json=body,timeout=45,allow_redirects=False)\n            value=response.json()\n        except Exception:raise ResearchError(\'COORDINATOR_UNREACHABLE\') from None\n        if response.status_code!=200:raise ResearchError(value.get(\'code\',\'COORDINATOR_FAILED\'),response.status_code)\n        return value\n    def huggingface(self,ticket):\n        repo=os.environ.get(\'NEBULA_HF_REPO\',\'\')\n        token=os.environ.get(\'NEBULA_HF_TOKEN\',\'\')\n        if (not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or ticket.get(\'repo\')!=repo or\n                not re.fullmatch(r\'checkpoints/[a-f0-9-]{36}\\.pt\',ticket.get(\'path\',\'\')) or\n                not token.startswith(\'hf_\')):\n            raise ResearchError(\'HF_CONFIGURATION_MISSING\')\n        try:\n            from huggingface_hub import HfApi\n            hub=HfApi(endpoint=\'https://huggingface.co\',token=token)\n            if hub.model_info(repo).private is not True:raise ResearchError(\'HF_REPOSITORY_MUST_BE_PRIVATE\')\n            return hub,repo\n        except ResearchError:raise\n        except Exception:raise ResearchError(\'HF_ACCESS_OR_DEPENDENCY_FAILED\') from None\n    def download(self,identifier,destination):\n        ticket=self.call(\'download\',{\'id\':identifier})\n        if ticket.get(\'provider\')==\'huggingface\':return self.download_huggingface(ticket,destination)\n        url=storage_url(ticket[\'url\'])\n        path=Path(destination);path.parent.mkdir(parents=True,exist_ok=True)\n        if path.exists() and path.stat().st_size==ticket[\'bytes\'] and digests(path)[0]==ticket[\'sha256\']:return path\n        temporary=path.with_suffix(path.suffix+\'.download-tmp\');sha=hashlib.sha256();count=0\n        try:\n            with requests.get(url,stream=True,timeout=90,allow_redirects=False) as response:\n                if response.status_code!=200:raise ResearchError(\'CHECKPOINT_DOWNLOAD_FAILED\')\n                with temporary.open(\'wb\') as stream:\n                    for chunk in response.iter_content(1024*1024):\n                        count+=len(chunk)\n                        if count>ticket[\'bytes\'] or count>100*1024*1024:raise ResearchError(\'CHECKPOINT_SIZE_MISMATCH\')\n                        sha.update(chunk);stream.write(chunk)\n            if count!=ticket[\'bytes\'] or sha.hexdigest()!=ticket[\'sha256\']:raise ResearchError(\'CHECKPOINT_CHECKSUM_MISMATCH\')\n            temporary.replace(path);return path\n        except ResearchError:raise\n        except Exception:raise ResearchError(\'CHECKPOINT_DOWNLOAD_FAILED\') from None\n    def download_huggingface(self,ticket,destination):\n        hub,repo=self.huggingface(ticket)\n        if (not re.fullmatch(r\'[a-f0-9]{40}\',ticket.get(\'revision\',\'\')) or\n                not isinstance(ticket.get(\'bytes\'),int) or not 0<ticket[\'bytes\']<=100*1024*1024):\n            raise ResearchError(\'HF_INVALID_DOWNLOAD\')\n        path=Path(destination);path.parent.mkdir(parents=True,exist_ok=True)\n        if path.exists() and path.stat().st_size==ticket[\'bytes\'] and digests(path)[0]==ticket[\'sha256\']:return path\n        temporary=path.with_suffix(path.suffix+\'.download-tmp\')\n        try:\n            from huggingface_hub import hf_hub_download\n            cached=Path(hf_hub_download(repo_id=repo,filename=ticket[\'path\'],revision=ticket[\'revision\'],\n                                       token=os.environ[\'NEBULA_HF_TOKEN\'],endpoint=\'https://huggingface.co\'))\n            if cached.stat().st_size!=ticket[\'bytes\'] or digests(cached)[0]!=ticket[\'sha256\']:\n                raise ResearchError(\'CHECKPOINT_CHECKSUM_MISMATCH\')\n            shutil.copyfile(cached,temporary);temporary.replace(path);return path\n        except ResearchError:raise\n        except Exception:raise ResearchError(\'HF_DOWNLOAD_FAILED\') from None\n    def upload(self,path,kind=\'checkpoint\',parent=None,lease=None):\n        path=Path(path);sha,md5=digests(path)\n        ticket=self.call(\'upload\',{\'kind\':kind,\'bytes\':path.stat().st_size,\'sha256\':sha,\'md5\':md5,\n                                   \'parent\':parent,\'lease\':lease})\n        if ticket.get(\'provider\')==\'huggingface\':\n            hub,repo=self.huggingface(ticket)\n            try:\n                hub.upload_file(path_or_fileobj=str(path),path_in_repo=ticket[\'path\'],repo_id=repo,\n                                commit_message=\'Save private Nebula research checkpoint\')\n            except Exception:\n                try:self.call(\'finish\',{\'id\':ticket[\'id\']})\n                except Exception:raise ResearchError(\'HF_UPLOAD_FAILED\') from None\n            else:self.call(\'finish\',{\'id\':ticket[\'id\']})\n            return ticket[\'id\']\n        try:\n            with path.open(\'rb\') as stream:\n                response=requests.put(storage_url(ticket[\'url\']),data=stream,headers=ticket[\'headers\'],timeout=180,allow_redirects=False)\n                # A timeout or retry may occur after an upload succeeded. The\n                # coordinator checks the immutable object before accepting it.\n                if response.status_code not in [200,201,412]:raise ResearchError(\'CHECKPOINT_UPLOAD_FAILED\')\n        except Exception:\n            try:self.call(\'finish\',{\'id\':ticket[\'id\']})\n            except Exception:raise ResearchError(\'CHECKPOINT_UPLOAD_FAILED\') from None\n        else:self.call(\'finish\',{\'id\':ticket[\'id\']})\n        return ticket[\'id\']\n', 'v18_decode_probe.py': '"""Inference-only comparison; never updates V18 weights, optimizer or manifest."""\nimport gc\nimport json\nimport os\nimport re\nimport uuid\nfrom collections import Counter\nfrom datetime import datetime, timezone\n\nimport requests\nimport torch\nfrom nebula import BOS, EOS, pack_prompt\nfrom v18_stream import Hub, deserialize, PROBES\n\nMODES = [dict(name=\'greedy\', temperature=0., top_p=1., seeds=[0]),\n         dict(name=\'sample-0.7\', temperature=.7, top_p=.9, seeds=[17,43]),\n         dict(name=\'sample-1.0\', temperature=1., top_p=.95, seeds=[17,43])]\nQUESTIONS = [\'Hello! What can you help me with?\', \'Explain why ice melts.\',\n             \'Write a thank-you note to a neighbor.\', \'Which should I choose?\',\n             \'My name is Iris. My scarf is purple. What color is my scarf?\']\n\n\ndef repetition(text):\n    words = re.findall(r"\\w+",text.casefold())\n    grams = [tuple(words[i:i+3]) for i in range(max(0,len(words)-2))]\n    counts = Counter(grams)\n    return {\'words\':len(words), \'repeatedTrigramFraction\':\n            round(sum(n-1 for n in counts.values())/len(grams),4) if grams else 0.,\n            \'utf8ReplacementCharacters\':text.count(\'\\ufffd\')}\n\n\n@torch.inference_mode()\ndef generate(model, prompt, kind, temperature, top_p, seed, max_bytes=128):\n    if kind not in [\'raw\',\'chat\'] or not 0<=temperature<=2 or not 0<top_p<=1:\n        raise ValueError(\'Invalid generation configuration\')\n    prefix = ([BOS]+list(prompt.encode(\'utf-8\'))) if kind==\'raw\' else pack_prompt(\n        [{\'role\':\'system\',\'content\':\'You are Nebula AI.\'},{\'role\':\'user\',\'content\':prompt}],model.config.context)\n    if len(prefix)>=model.config.context:raise ValueError(\'Prompt leaves no context for generation\')\n    device=next(model.parameters()).device\n    rng=torch.Generator(device=device).manual_seed(seed)\n    tokens=torch.tensor([prefix],dtype=torch.long,device=device)\n    generated=[];stop=\'byte-budget\';model.eval()\n    limit=min(max_bytes,model.config.context-len(prefix))\n    for _ in range(limit):\n        logits=model(tokens)[0,-1].float().clone()\n        # Match the original V18 raw probe\'s stop-on-any-marker rule.\n        # Chat follows Nebula.reply: only EOS may be generated among markers.\n        if kind==\'chat\':\n            end=logits[EOS].clone();logits[BOS:]=float(\'-inf\');logits[EOS]=end\n        if temperature==0:\n            value=int(logits.argmax())\n        else:\n            values,indices=torch.sort(logits/temperature,descending=True)\n            probs=torch.softmax(values,dim=-1)\n            remove=probs.cumsum(-1)-probs>=top_p\n            values[remove]=float(\'-inf\')\n            sampled=torch.multinomial(torch.softmax(values,dim=-1),1,generator=rng)\n            value=int(indices[sampled].item())\n        if value>=256:\n            stop=\'eos\' if value==EOS else \'special-marker\';break\n        generated.append(value)\n        tokens=torch.cat([tokens,torch.tensor([[value]],device=device)],dim=1)\n    else:\n        if limit<max_bytes:stop=\'context-limit\'\n    text=bytes(generated).decode(\'utf-8\',errors=\'replace\')\n    return {\'reply\':text,\'generatedBytes\':len(generated),\'stopReason\':stop,\n            **repetition(text),\'relevanceGrade\':\'pending-human-review\'}\n\n\ndef compare(model, max_bytes=128, modes=MODES, raw=PROBES, questions=QUESTIONS):\n    answers=[]\n    total=(len(raw)+len(questions))*sum(len(m[\'seeds\']) for m in modes)\n    for kind,prompts in [(\'raw\',raw),(\'chat\',questions)]:\n        for prompt in prompts:\n            for mode in modes:\n                for seed in mode[\'seeds\']:\n                    result=generate(model,prompt,kind,mode[\'temperature\'],mode[\'top_p\'],seed,max_bytes)\n                    answer={\'kind\':kind,\'prompt\':prompt,\'mode\':mode[\'name\'],\'seed\':seed,\n                            \'temperature\':mode[\'temperature\'],\'topP\':mode[\'top_p\'],**result}\n                    answers.append(answer)\n                    print(json.dumps({\'decodeProbeProgress\':len(answers),\'total\':total,**answer}),flush=True)\n    summaries=[]\n    for kind in [\'raw\',\'chat\']:\n        for mode in modes:\n            rows=[a for a in answers if a[\'kind\']==kind and a[\'mode\']==mode[\'name\']]\n            if rows:summaries.append({\'kind\':kind,\'mode\':mode[\'name\'],\'cases\':len(rows),\n                \'meanRepeatedTrigramFraction\':sum(a[\'repeatedTrigramFraction\'] for a in rows)/len(rows),\n                \'emptyReplies\':sum(not a[\'reply\'].strip() for a in rows),\n                \'notice\':\'Less repetition does not establish correctness or relevance.\'})\n    return {\'answers\':answers,\'summary\':summaries,\'qualityApproved\':False,\'humanVerified\':False,\n            \'trainingStarted\':False,\'publicWebsiteUpdated\':False}\n\n\ndef run_probe(repo=\'iiidevv/nebula-raw-training\',run_name=\'nebula-v18-raw-1gb\',\n              revision=\'e167fff3bb27c36c25460fcd4c61915a5b7016ed\',expected_updates=10102,\n              save_report=True,max_bytes=128):\n    if not torch.cuda.is_available():raise RuntimeError(\'Enable a Kaggle NVIDIA GPU before running this probe.\')\n    if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or not re.fullmatch(r\'nebula-v18-[\\w-]{1,45}\',run_name):\n        raise ValueError(\'Invalid repository/run\')\n    token=os.environ.get(\'NEBULA_HF_TOKEN\') or os.environ.get(\'HF_TOKEN\')\n    if not token:raise RuntimeError(\'Enable Kaggle secret NEBULA_HF_TOKEN.\')\n    from huggingface_hub import HfApi,CommitOperationAdd\n    # Construct a reader without Hub.__init__: no repository creation or manifest writes.\n    hub=object.__new__(Hub);hub.repo=repo;hub.token=token;hub.prefix=\'experiments/\'+run_name+\'/\'\n    hub.api=HfApi(token=token);hub.session=requests.Session()\n    hub.revision=hub.api.model_info(repo,revision=revision or None).sha\n    state=hub.state()\n    if not state or not state.get(\'working\'):raise RuntimeError(\'Saved working checkpoint is missing.\')\n    if expected_updates is not None and state.get(\'updates\')!=expected_updates:\n        raise RuntimeError(\'Checkpoint step differs from EXPECTED_UPDATES; verify the revision before testing.\')\n    model,payload=deserialize(hub.read_entry(state[\'working\']))\n    del payload;gc.collect()\n    torch.set_num_threads(2);model.to(\'cuda\').eval()\n    print(json.dumps({\'decodeProbeCheckpoint\':state[\'updates\'],\'revision\':hub.revision,\n        \'weightObjectSha256\':state[\'working\'][\'sha256\'],\'GPU\':torch.cuda.get_device_name(0),\n        \'precision\':\'float32\',\'trainingStarted\':False}),flush=True)\n    report=compare(model,max_bytes)\n    report.update(format=\'nebula-v18-decoding-comparison-1\',checkpointUpdates=state[\'updates\'],\n        checkpointRevision=hub.revision,weightObjectSha256=state[\'working\'][\'sha256\'],\n        generatedAt=datetime.now(timezone.utc).isoformat(),maxGeneratedBytes=max_bytes,\n        notice=\'Synthetic inference probes only. No weight/optimizer updates. No byte repetition penalty applied.\')\n    if save_report:\n        name=\'decode-\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')+\'-\'+uuid.uuid4().hex[:8]+\'.json\'\n        path=\'experiments/\'+run_name+\'/decode-reports/\'+name\n        commit=hub.api.create_commit(repo_id=repo,operations=[CommitOperationAdd(\n            path_in_repo=path,path_or_fileobj=json.dumps(report,indent=2).encode())],\n            commit_message=\'Save inference-only decoding comparison\')\n        report[\'savedReportUrl\']=\'https://huggingface.co/\'+repo+\'/blob/\'+commit.oid+\'/\'+path\n        print(json.dumps({\'decodeReportSaved\':True,\'url\':report[\'savedReportUrl\'],\n                          \'trainingManifestModified\':False}),flush=True)\n    del model;gc.collect();torch.cuda.empty_cache();hub.session.close()\n    return report\n', 'v18_foundation_control.py': '"""Audit actual saved raw batches and run a bounded story foundation experiment."""\nimport copy\nimport gc\nimport gzip\nimport json\nimport math\nimport os\nimport random\nimport time\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport requests\nimport torch\nimport torch.nn.functional as F\nfrom nebula import BOS,EOS\nfrom v18_stream import Hub,RemotePool,deserialize,serialize,sha,digest,emit,PROBES\nfrom v18_decode_probe import compare\n\nSOURCE_REVISION=\'e167fff3bb27c36c25460fcd4c61915a5b7016ed\'\nDEFAULT={\'sourceRepo\':\'iiidevv/nebula-raw-training\',\'sourceRun\':\'nebula-v18-raw-1gb\',\n         \'sourceRevision\':SOURCE_REVISION,\'expectedUpdates\':10102,\'updates\':1000,\n         \'trainUtf8Bytes\':8_000_000,\'validationUtf8Bytes\':500_000,\'seed\':19021,\n         \'saveEvery\':100,\'maxSessionHours\':2}\n\n\ndef code_signature():\n    root=Path(__file__).parent\n    return sha(\'\'.join((root/n).read_text(encoding=\'utf-8\') for n in\n               [\'v18_foundation_control.py\',\'nebula.py\',\'v18_stream.py\',\'v18_decode_probe.py\']).encode())\n\n\ndef read_source(repo,run,revision,token):\n    from huggingface_hub import HfApi\n    hub=object.__new__(Hub);hub.repo=repo;hub.prefix=\'experiments/\'+run+\'/\'\n    hub.api=HfApi(token=token);hub.token=token;hub.session=requests.Session()\n    hub.revision=hub.api.model_info(repo,revision=revision).sha\n    state=hub.state()\n    if not state or not state.get(\'working\') or not state.get(\'corpusComplete\'):\n        raise ValueError(\'The pinned source needs a working checkpoint and completed corpus.\')\n    return hub,state\n\n\ndef load_index(hub,state):\n    records=[];seen=set();counts={};sizes={};split_counts={\'train\':0,\'validation\':0}\n    spans={};previous={}\n    for entry in state[\'corpus\'][\'index\']:\n        for line in gzip.decompress(hub.read_entry(entry)).splitlines():\n            record=json.loads(line);identifier=record[\'hash\'];split=record[\'split\']\n            if split not in split_counts:raise ValueError(\'Invalid document split\')\n            if identifier in seen:raise ValueError(\'Duplicate normalized document hash or split overlap\')\n            if (\'validation\' if int(identifier[:8],16)%100<state[\'config\'][\'validationPercent\'] else \'train\')!=split:\n                raise ValueError(\'Document hash split differs from the training preparation rule\')\n            shard=record[\'shard\'];start=record[\'startToken\'];end=record[\'endToken\']\n            if not 0<=shard<len(state[\'corpus\'][split]):raise ValueError(\'Invalid shard index\')\n            if not 0<=start<end<=state[\'corpus\'][split][shard][\'bytes\']//2:\n                raise ValueError(\'Document range outside its saved shard\')\n            if end-start!=record[\'utf8Bytes\']+2:raise ValueError(\'Document length differs from byte count plus markers\')\n            key=(split,shard)\n            if previous.get(key,0)!=start:raise ValueError(\'Gap or overlap in indexed packed documents\')\n            previous[key]=end;spans[key]=end\n            seen.add(identifier);source=record[\'source\'];counts[source]=counts.get(source,0)+1\n            sizes[source]=sizes.get(source,0)+record[\'utf8Bytes\'];split_counts[split]+=1;records.append(record)\n        emit(controlAuditIndexDocuments=len(records))\n    for split in split_counts:\n        for i,entry in enumerate(state[\'corpus\'][split]):\n            if spans.get((split,i),0)!=entry[\'bytes\']//2:raise ValueError(\'Index does not cover the entire saved shard\')\n    if sum(sizes.values())!=state[\'uniqueTextBytes\'] or sizes!=state[\'sourceBytes\']:\n        raise ValueError(\'Indexed source totals differ from the saved corpus manifest\')\n    if len(records)!=state[\'documents\']:raise ValueError(\'Indexed document count differs from manifest\')\n    return records,{\'documents\':len(records),\'sourceDocuments\':counts,\'sourceUtf8Bytes\':sizes,\n        \'splitDocuments\':split_counts,\'uniqueNormalizedHashes\':len(seen),\n        \'indexCoverageComplete\':True,\'trainValidationHashesDisjoint\':True,\n        \'scope\':\'All index metadata checked; payload integrity/UTF-8/content checked on samples and selected story documents.\'}\n\n\ndef document(pool,record):\n    # Pool entries must retain original order: constructor filters undersized shards.\n    values=pool.data(record[\'shard\'])[record[\'startToken\']:record[\'endToken\']]\n    if len(values)!=record[\'utf8Bytes\']+2 or values[0]!=BOS or values[-1]!=EOS:\n        raise ValueError(\'Saved document boundary/length mismatch\')\n    interior=values[1:-1]\n    if np.any(interior>255):raise ValueError(\'Unexpected role/special token inside raw document\')\n    content=interior.astype(np.uint8).tobytes();text=content.decode(\'utf-8\',errors=\'strict\')\n    identifier=sha(\' \'.join(unicodedata.normalize(\'NFKC\',text).casefold().split()).encode())\n    if identifier!=record[\'hash\']:raise ValueError(\'Payload document hash differs from source index\')\n    return values,text\n\n\ndef audit_payloads(hub,state,records,context):\n    # No shard filtering here: an end shard can be shorter than context.\n    pools={}\n    for split in [\'train\',\'validation\']:\n        pool=object.__new__(RemotePool);pool.hub=hub;pool.entries=state[\'corpus\'][split]\n        from collections import OrderedDict\n        pool.cache=OrderedDict();pools[split]=pool\n    samples=[]\n    for source in sorted(state[\'sourceBytes\']):\n        for split in [\'train\',\'validation\']:\n            rows=sorted((r for r in records if r[\'source\']==source and r[\'split\']==split),key=lambda r:r[\'hash\'])[:4]\n            for row in rows:\n                _,text=document(pools[split],row)\n                samples.append({\'source\':source,\'split\':split,\'hash\':row[\'hash\'],\'utf8Bytes\':row[\'utf8Bytes\'],\n                                \'textBeginning\':text[:500]})\n    # Reconstruct the real V18 random batch at the saved optimizer update.\n    original=RemotePool(hub,state[\'corpus\'][\'train\'],context,state[\'config\'][\'shardHoldSteps\'])\n    step=state[\'updates\'];seed=state[\'config\'][\'seed\'];batch=state[\'config\'][\'batch\']\n    chosen=random.Random(seed+step//original.hold).choices(range(len(original.entries)),weights=original.weights,k=1)[0]\n    data=original.data(chosen);rng=random.Random(seed+step*7919)\n    pairs=original.batch(step,batch,seed);windows=[]\n    for x,y in pairs:\n        start=rng.randrange(len(data)-context)\n        expected=torch.tensor(np.array(data[start:start+context+1],dtype=np.int64))\n        if not torch.equal(x,expected[:-1]) or not torch.equal(y,expected[1:]) or not torch.equal(x[1:],y[:-1]):\n            raise ValueError(\'Actual V18 input/target alignment failed\')\n        if int(x.min())<0 or int(y.min())<0 or int(x.max())>EOS or int(y.max())>EOS:\n            raise ValueError(\'Unexpected token in raw training batch\')\n        windows.append({\'startToken\':start,\'inputLength\':len(x),\'targetLength\':len(y),\n                        \'bosTargets\':int((y==BOS).sum()),\'eosTargets\':int((y==EOS).sum()),\n                        \'decodedBeginning\':bytes(int(v) for v in x[:300] if v<256).decode(\'utf-8\',errors=\'replace\')})\n    result={\'payloadSamplesChecked\':len(samples),\'samples\':samples,\'actualV18Batch\':{\n        \'reconstructedUpdate\':step,\'batch\':batch,\'context\':context,\'targetShiftCorrect\':True,\'windows\':windows},\n        \'notice\':\'A window beginning mid-word or cutting a UTF-8 character is expected in byte-level packed training. Samples are not a full semantic/PII audit.\'}\n    del original,data\n    for pool in pools.values():pool.cache.clear()\n    gc.collect()\n    return pools,result\n\n\ndef select_stories(pools,records,train_limit,valid_limit):\n    arrays={};selected={};prefixes=[]\n    for split,limit in [(\'train\',train_limit),(\'validation\',valid_limit)]:\n        packed=bytearray();hashes=[];total=0\n        rows=sorted((r for r in records if r[\'source\']==\'stories\' and r[\'split\']==split),key=lambda r:r[\'hash\'])\n        # Read selected records in shard order to avoid re-downloading the same shard for each hash.\n        chosen=[]\n        for r in rows:\n            if total+r[\'utf8Bytes\']>limit:continue\n            chosen.append(r);total+=r[\'utf8Bytes\']\n            if total>=limit*.995:break\n        for r in sorted(chosen,key=lambda r:(r[\'shard\'],r[\'startToken\'])):\n            values,text=document(pools[split],r);packed.extend(values.tobytes());hashes.append(r[\'hash\'])\n            if split==\'validation\' and len(prefixes)<2:\n                prefix=text[:85].rsplit(\' \',1)[0]\n                if prefix:prefixes.append(prefix)\n        if not packed:raise ValueError(\'No indexed TinyStories documents available in \'+split)\n        arrays[split]=np.frombuffer(packed,dtype=\'<u2\')\n        selected[split]={\'documents\':len(hashes),\'utf8Bytes\':total,\'packedSha256\':sha(packed),\'hashes\':hashes}\n        pools[split].cache.clear()\n    if set(selected[\'train\'][\'hashes\'])&set(selected[\'validation\'][\'hashes\']):raise ValueError(\'Story split leakage\')\n    return arrays,selected,prefixes\n\n\ndef pairs_from_array(data,context,batch,seed):\n    if len(data)<=context:raise ValueError(\'Story pool too short for full context\')\n    rng=random.Random(seed);pairs=[]\n    for _ in range(batch):\n        start=rng.randrange(len(data)-context)\n        values=torch.tensor(np.array(data[start:start+context+1],dtype=np.int64))\n        pairs.append((values[:-1],values[1:]))\n    return pairs\n\n\n@torch.inference_mode()\ndef heldout(model,pairs):\n    model.eval();device=next(model.parameters()).device;total=0.;targets=0\n    for i,(x,y) in enumerate(pairs):\n        loss=F.cross_entropy(model(x[None].to(device)).float().flatten(0,1),y.to(device),reduction=\'sum\')\n        if not torch.isfinite(loss):raise ValueError(\'Non-finite held-out loss\')\n        total+=float(loss);targets+=len(y)\n        if (i+1)%8==0:emit(controlHeldoutBlocks=i+1,total=len(pairs))\n    return {\'loss\':total/targets,\'targets\':targets}\n\n\n@torch.inference_mode()\ndef audit_forward(model,pair):\n    model.eval();device=next(model.parameters()).device\n    x,y=(v[:96].to(device) for v in pair)\n    before=model(x[None]).float()\n    changed=x.clone();cut=len(x)//2;changed[cut:]=(changed[cut:]+17)%256\n    after=model(changed[None]).float()\n    if not torch.isfinite(before).all() or not torch.isfinite(after).all():raise ValueError(\'Non-finite source logits\')\n    delta=float((before[:,:cut]-after[:,:cut]).abs().max())\n    if not torch.allclose(before[:,:cut],after[:,:cut],rtol=1e-4,atol=1e-4):\n        raise ValueError(\'Future-token changes affected earlier logits on a real saved batch\')\n    batched=F.cross_entropy(before.flatten(0,1),y)\n    reference=-torch.log_softmax(before[0],-1)[torch.arange(len(y),device=device),y].mean()\n    if not torch.allclose(batched,reference,rtol=1e-5,atol=1e-5):raise ValueError(\'Reference causal loss mismatch\')\n    return {\'futureChangeMaxPastLogitDelta\':delta,\'causalPrefixInvariant\':True,\n            \'manualNextTokenLossMatches\':True,\'sampleLoss\':float(batched),\'positions\':len(y)}\n\n\ndef run_control(output_run=\'nebula-v18-foundation-control\',config=None,device=\'cuda\',clock=None):\n    clock=time.monotonic if clock is None else clock\n    config=copy.deepcopy(DEFAULT if config is None else config)\n    if set(config)!=set(DEFAULT) or not 1<=config[\'updates\']<=5000 or not 0<config[\'maxSessionHours\']<=8:\n        raise ValueError(\'Invalid controlled experiment configuration\')\n    if output_run==config[\'sourceRun\']:raise ValueError(\'Use a different run; the source must be preserved\')\n    if device==\'cuda\' and not torch.cuda.is_available():raise RuntimeError(\'Enable a Kaggle NVIDIA GPU\')\n    token=os.environ.get(\'NEBULA_HF_TOKEN\') or os.environ.get(\'HF_TOKEN\')\n    if not token:raise RuntimeError(\'Enable Kaggle secret NEBULA_HF_TOKEN\')\n    source,source_state=read_source(config[\'sourceRepo\'],config[\'sourceRun\'],config[\'sourceRevision\'],token)\n    if source_state[\'updates\']!=config[\'expectedUpdates\']:raise ValueError(\'Pinned source step mismatch\')\n    output=Hub(config[\'sourceRepo\'],token,output_run);state=output.state();signature=digest({\'config\':config,\'code\':code_signature()})\n    if state and state.get(\'signature\')!=signature:raise ValueError(\'Keep the original experiment settings/code to resume; otherwise choose a NEW output run\')\n    if state and state.get(\'complete\'):\n        emit(controlAlreadyComplete=True,updates=state[\'updates\'],publicWebsiteUpdated=False)\n        return json.loads(output.read_entry(state[\'finalReport\']))\n    if state is None:\n        state={\'format\':\'nebula-v18-foundation-control-1\',\'signature\':signature,\'config\':config,\n            \'sourceRevision\':source.revision,\'sourceWeightSha256\':source_state[\'working\'][\'sha256\'],\n            \'sourceUpdates\':source_state[\'updates\'],\'updates\':0,\'complete\':False,\'qualityApproved\':False,\'publicWebsiteUpdated\':False}\n    records,index_audit=load_index(source,source_state)\n    model,payload=deserialize(output.read_entry(state[\'working\']) if state.get(\'working\') else source.read_entry(source_state[\'working\']))\n    context=model.config.context\n    pools,payload_audit=audit_payloads(source,source_state,records,context)\n    arrays,selection,extra_prefixes=select_stories(pools,records,config[\'trainUtf8Bytes\'],config[\'validationUtf8Bytes\'])\n    selection_hash=digest(selection)\n    if state.get(\'selectionHash\') and state[\'selectionHash\']!=selection_hash:raise ValueError(\'Reconstructed story selection changed\')\n    model.to(device);torch.set_num_threads(2)\n    if not state.get(\'audit\'):\n        actual_pair=RemotePool(source,source_state[\'corpus\'][\'train\'],context,source_state[\'config\'][\'shardHoldSteps\']).batch(\n            source_state[\'updates\'],1,source_state[\'config\'][\'seed\'])[0]\n        forward_audit=audit_forward(model,actual_pair)\n    else:forward_audit={\'notice\':\'Original source forward audit already saved with this experiment.\'}\n    audit={\'index\':index_audit,\'payload\':payload_audit,\'storySelection\':selection,\'forward\':forward_audit,\n           \'passed\':True,\'humanVerified\':False,\'scope\':\'Packing/alignment/integrity checks, not proof of semantic correctness or full model architecture.\'}\n    emit(controlAuditSummary=index_audit,forwardAudit=forward_audit,\n         actualBatchChecks={k:v for k,v in payload_audit[\'actualV18Batch\'].items() if k!=\'windows\'})\n    seen_samples=set()\n    for sample in payload_audit[\'samples\']:\n        key=(sample[\'source\'],sample[\'split\'])\n        if key not in seen_samples:\n            emit(controlPublicDocumentSample=sample);seen_samples.add(key)\n    for window in payload_audit[\'actualV18Batch\'][\'windows\'][:2]:emit(controlActualWindow=window)\n    if not state.get(\'audit\'):\n        state=output.publish({**state,\'selectionHash\':selection_hash},{\'audit\':(\'audit.json\',json.dumps(audit,indent=2).encode())})\n    emit(controlAuditPassed=True,storyTrainBytes=selection[\'train\'][\'utf8Bytes\'],storyValidationBytes=selection[\'validation\'][\'utf8Bytes\'])\n    del records,pools,audit,payload_audit;gc.collect()\n    amp=device==\'cuda\';native_bf16=amp and torch.cuda.get_device_capability(0)[0]>=8 and torch.cuda.is_bf16_supported()\n    dtype=torch.bfloat16 if native_bf16 else torch.float16\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=amp and not native_bf16,init_scale=1024)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=source_state[\'config\'][\'learningRate\'],weight_decay=.01)\n    training=payload[\'trainingState\'];optimizer.load_state_dict(training[\'optimizer\'])\n    if state.get(\'working\'):\n        if training.get(\'controlSignature\')!=signature:raise ValueError(\'Controlled checkpoint signature mismatch\')\n        if training[\'updates\']!=state[\'updates\']:raise ValueError(\'Controlled checkpoint step mismatch\')\n    if scaler.is_enabled() and training.get(\'scaler\'):scaler.load_state_dict(training[\'scaler\'])\n    del payload,training;gc.collect()\n    mixed=RemotePool(source,source_state[\'corpus\'][\'validation\'],context,1)\n    mixed_pairs=[mixed.batch(i,1,880019)[0] for i in range(16)]\n    story_pairs=pairs_from_array(arrays[\'validation\'],context,16,config[\'seed\']+1)\n    del mixed;gc.collect()\n    modes=[dict(name=\'greedy\',temperature=0.,top_p=1.,seeds=[0]),\n           dict(name=\'sample-0.7\',temperature=.7,top_p=.9,seeds=[17,43])]\n    def evaluate(label):\n        emit(controlEvaluation=label)\n        report={\'stage\':label,\'additionalUpdates\':state[\'updates\'],\n                \'heldoutMixed\':heldout(model,mixed_pairs),\'heldoutStories\':heldout(model,story_pairs),\n                **compare(model,128,modes=modes,raw=PROBES+extra_prefixes)}\n        # compare\'s generic inference-only flag refers to the probe, not this whole experiment.\n        report[\'trainingStarted\']=state[\'updates\']>0\n        return report\n    if not state.get(\'baselineReport\'):\n        baseline=evaluate(\'baseline-10102\')\n        state=output.publish(state,{\'baselineReport\':(\'baseline.json\',json.dumps(baseline,indent=2).encode())})\n    started=clock();bad=0;last_saved=state[\'updates\'];interrupted=False\n    source_config=source_state[\'config\'];batch=source_config[\'batch\'];microbatch=source_config[\'microbatch\']\n    emit(controlTraining=True,startingStep=source_state[\'updates\'],additionalUpdateBudget=config[\'updates\'],\n         optimizerRestored=True,precision=str(dtype) if amp else \'float32\',trainingGPUs=1,\n         change=\'Train only on a deterministic public TinyStories subset; retain model, batch and original learning-rate schedule.\',\n         limitedChange=\'T4 uses native FP16 AMP instead of emulated BF16 when inherited source used it.\',publicWebsiteUpdated=False)\n    def save():\n        nonlocal state,last_saved\n        stream=serialize(model,{\'experimental\':True,\'qualityApproved\':False,\'sourceUpdates\':source_state[\'updates\'],\n                     \'additionalUpdates\':state[\'updates\']},\n            {\'controlSignature\':signature,\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\'updates\':state[\'updates\']})\n        try:state=output.publish(state,{\'working\':(\'working.pt\',stream.getbuffer())})\n        finally:stream.close()\n        last_saved=state[\'updates\']\n        emit(controlCheckpointConfirmed=True,additionalUpdates=last_saved,publicWebsiteUpdated=False)\n    try:\n        while state[\'updates\']<config[\'updates\'] and clock()-started<config[\'maxSessionHours\']*3600:\n            step=state[\'updates\'];absolute=source_state[\'updates\']+step\n            rate=source_config[\'learningRate\']*min(1.,(absolute+1)/source_config[\'warmup\'])*(.1+.9*.5*(1+math.cos(math.pi*absolute/source_config[\'steps\'])))\n            for group in optimizer.param_groups:group[\'lr\']=rate\n            model.train();optimizer.zero_grad(set_to_none=True);loss_value=0.\n            pairs=pairs_from_array(arrays[\'train\'],context,batch,config[\'seed\']+step*7919)\n            for offset in range(0,batch,microbatch):\n                micro=pairs[offset:offset+microbatch]\n                x=torch.stack([p[0] for p in micro]).to(device);y=torch.stack([p[1] for p in micro]).to(device)\n                with torch.autocast(\'cuda\',dtype=dtype,enabled=amp):\n                    loss=F.cross_entropy(model(x).flatten(0,1),y.flatten(),reduction=\'sum\')/(batch*context)\n                if not torch.isfinite(loss):raise ValueError(\'Non-finite loss; keep the last confirmed control checkpoint\')\n                loss_value+=float(loss.detach());scaler.scale(loss).backward()\n            scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            if not torch.isfinite(norm):\n                bad+=1\n                if not scaler.is_enabled() or bad>12:raise ValueError(\'Non-finite gradients\')\n                scaler.update(new_scale=scaler.get_scale()/2);continue\n            scaler.step(optimizer);scaler.update();bad=0;state[\'updates\']+=1\n            if state[\'updates\']%25==0:\n                emit(controlUpdate=state[\'updates\'],budget=config[\'updates\'],loss=round(loss_value,4),\n                     learningRate=rate,trainingBytePositions=state[\'updates\']*batch*context)\n            if state[\'updates\']%config[\'saveEvery\']==0:save()\n    except KeyboardInterrupt:interrupted=True\n    if state[\'updates\']!=last_saved:save()\n    if interrupted or state[\'updates\']<config[\'updates\']:\n        emit(controlPaused=True,updatesSaved=last_saved,runAgainWithSameSettings=True)\n        return state\n    final=evaluate(\'story-foundation-final\')\n    baseline=json.loads(output.read_entry(state[\'baselineReport\']))\n    report={\'format\':\'nebula-foundation-control-comparison-1\',\'sourceRevision\':source.revision,\n            \'sourceWeightSha256\':source_state[\'working\'][\'sha256\'],\'baseline\':baseline,\'final\':final,\n            \'additionalUpdates\':state[\'updates\'],\'additionalTrainingPositions\':state[\'updates\']*batch*context,\n            \'qualityApproved\':False,\'humanVerified\':False,\'publicWebsiteUpdated\':False,\n            \'interpretation\':\'Compare actual completions and both held-out losses. Repetition reductions alone do not establish coherence. No automatic promotion or source replacement.\'}\n    state=output.publish({**state,\'complete\':True},{\'finalReport\':(\'comparison.json\',json.dumps(report,indent=2).encode())})\n    emit(controlFinished=True,updatesSaved=state[\'updates\'],sourceWeightsPreserved=True,publicWebsiteUpdated=False,\n         reportParts=[p[\'path\'] for p in state[\'finalReport\'][\'parts\']])\n    del optimizer,model;gc.collect()\n    if amp:torch.cuda.empty_cache()\n    return report\n'}
import importlib
code_root=Path.cwd()/'nebula-foundation-control-code'
code_root.mkdir(exist_ok=True)
for name,source in sources.items():(code_root/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(code_root))
for name in sources:sys.modules.pop(Path(name).stem,None)
importlib.invalidate_caches()
from v18_foundation_control import run_control
comparison=run_control(OUTPUT_RUN,CONFIG)
if 'baseline' in comparison and 'final' in comparison:
    import pandas as pd
    from IPython.display import display,Markdown
    losses=[]
    for stage in ['baseline','final']:
        for pool in ['heldoutMixed','heldoutStories']:
            losses.append({'stage':stage,'pool':pool,**comparison[stage][pool]})
    display(pd.DataFrame(losses))
    for stage in ['baseline','final']:
        display(Markdown('## '+stage.title()))
        display(pd.DataFrame(comparison[stage]['summary']))
        for row in comparison[stage]['answers']:
            print(row['kind'],row['mode'],'seed',row['seed'])
            print('PROMPT:',row['prompt'])
            print('REPLY:',row['reply'])
            print()
else:
    print('Session paused. Run all again with the same configuration to continue.')
